# Example 2: Physical Room with White Gaussian Noise RIRs

This notebook shows how to create a **physical room** in PyRES using the `PhRoom_wgn` class.

The physical room represents the actual acoustic space where a Reverberation Enhancement System (RES) operates. Here we simulate the room impulse responses (RIRs) by generating exponentially decaying white Gaussian noise. This is a simple stochastic model that captures the reverberation time but not detailed modal behavior.

The room contains four types of transducers:
- **Stage emitters** (sound sources on stage)
- **System microphones** (microphones of the RES)
- **System loudspeakers** (loudspeakers of the RES)
- **Audience receivers** (listening positions in the audience area)

The class `PhRoom_wgn` automatically generates positions for these transducers (randomly within predefined zones) and computes the RIRs between each pair of groups.

Let’s begin.

## 1. Imports and Path Setup

Add the parent directory to the Python path so that PyRES can be imported (assuming the notebook is in the `examples/` folder).

In [ ]:
import sys
import os
# Add parent directory to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import matplotlib.pyplot as plt
from PyRES.physical_room import PhRoom_wgn

## 2. Time–Frequency Parameters

These parameters control the FFT processing and anti‑aliasing decay. They are used internally when converting the RIRs to the frequency domain (the RIRs themselves are generated in the time domain).

In [ ]:
samplerate = 48000          # Hz
nfft = samplerate * 3       # FFT size (3 seconds)
alias_decay_db = 0          # No extra anti‑aliasing decay

## 3. Physical Room Configuration

Define the room dimensions (length, width, height in meters), the desired reverberation time $T_{60}$ (in seconds), and the number of microphones and loudspeakers for the RES.

The `PhRoom_wgn` constructor will:
- Randomly place the stage emitters (one by default), system microphones, system loudspeakers, and audience receivers (one by default) inside the room.
- Generate RIRs between each pair of groups using exponentially decaying white noise with the given $T_{60}$.
- Store all RIRs as FLAMO `Filter` objects (time‑domain filters) that can later be used for frequency‑domain processing.

In [ ]:
room_dims = (12.1, 8.5, 3.2)   # length, width, height (m)
room_RT = 0.7                   # reverberation time (s)
n_M = 4                         # number of system microphones
n_L = 8                         # number of system loudspeakers

physical_room = PhRoom_wgn(
    fs=samplerate,
    nfft=nfft,
    alias_decay_db=alias_decay_db,
    room_dims=room_dims,
    room_RT=room_RT,
    n_M=n_M,
    n_L=n_L
)

## 4. Inspect the Physical Room

The class provides several attributes to examine the transducer setup and the generated RIRs.

In [ ]:
print(f"\nThe PhRoom_wgn class is a subclass of the {type(physical_room).__bases__[0].__name__} class.")
print(f"The physical room was created with {n_M} microphones and {n_L} loudspeakers, and the simulations were performed considering 1 stage emitter and 1 audience receiver.")
print("The information about all emitters and receivers are contained in the 'transducer_number' and 'transducer_positions' attributes, respectively.")

print("\nTransducer numbers:")
print(f"  Stage emitters: {physical_room.transducer_number['stg']}")
print(f"  System microphones: {physical_room.transducer_number['mcs']}")
print(f"  System loudspeakers: {physical_room.transducer_number['lds']}")
print(f"  Audience receivers: {physical_room.transducer_number['aud']}")

print("\nTransducer positions (x, y, z in meters):")
print(f"  Stage emitters: \n{physical_room.transducer_positions['stg']}")
print(f"  System microphones: \n{physical_room.transducer_positions['mcs']}")
print(f"  System loudspeakers: \n{physical_room.transducer_positions['lds']}")
print(f"  Audience receivers: \n{physical_room.transducer_positions['aud']}")

## 5. Visualise the Room Setup

The `plot_setup()` method generates a 3D scatter plot showing the positions of all transducers. This helps to verify the random placement.

In [ ]:
physical_room.plot_setup()
plt.show()

## 6. Examine the Room Impulse Responses

The RIRs are stored as FLAMO `Filter` objects in the attributes:
- `h_SA` : stage → audience
- `h_SM` : stage → microphones
- `h_LM` : loudspeakers → microphones
- `h_LA` : loudspeakers → audience

Each filter’s parameters are a tensor of shape `(num_receivers, num_sources, filter_length)` (or the transpose depending on convention). We can check their types and shapes.

In [ ]:
print("\nThe RIRs are contained in the 'h_SA', 'h_SM', 'h_LM', and 'h_LA' attributes:")
print(f"  stage → audience (h_SA): {type(physical_room.get_stg_to_aud())}, shape {physical_room.get_stg_to_aud().param.shape}")
print(f"  stage → microphones (h_SM): {type(physical_room.get_stg_to_mcs())}, shape {physical_room.get_stg_to_mcs().param.shape}")
print(f"  loudspeakers → microphones (h_LM): {type(physical_room.get_lds_to_mcs())}, shape {physical_room.get_lds_to_mcs().param.shape}")
print(f"  loudspeakers → audience (h_LA): {type(physical_room.get_lds_to_aud())}, shape {physical_room.get_lds_to_aud().param.shape}")

## 7. Conclusion

You have successfully created a physical room model with stochastic RIRs. The generated filters can now be used in combination with a virtual room (as in Example 1) to simulate the complete RES.

To explore further:
- Change the room dimensions or reverberation time.
- Increase the number of stage emitters or audience receivers (by passing `n_stg` and `n_aud` arguments to the constructor).
- Try other physical room classes (e.g., `PhRoom_ism` for image‑source method) if available.

For more details, refer to the documentation in `PyRES/physical_room.py`.